# MAESTRO Preprocessing (Updated with Supplementary Guide)

Includes fixes:
1. Use explicit Train/Val/Test splits from metadata (no random splitting).
2. Piano-roll (Tasks 1 & 2) window sparsity filtering (< 2% active cells discarded).
3. Token formulation using `miditok` (REMI scheme) for Tasks 3 & 4.

In [1]:
!pip install pretty_midi pandas matplotlib tqdm miditok


[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
import numpy as np
import pandas as pd
import pretty_midi
import matplotlib.pyplot as plt
from tqdm import tqdm
from miditok import REMI, TokenizerConfig
import warnings
warnings.filterwarnings('ignore')

c:\Users\lubdh\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import os
import urllib.request
import zipfile

raw_root = os.path.join("data", "raw_midi")
maestro_dir = os.path.join(raw_root, "maestro-v3.0.0")
fallback_dir = os.path.join("data", "maestro-v3.0.0")
metadata_file = "maestro-v3.0.0.csv"

if not os.path.exists(os.path.join(maestro_dir, metadata_file)) and not os.path.exists(os.path.join(fallback_dir, metadata_file)):
    print("Downloading MAESTRO dataset (MIDI only)...")
    os.makedirs(raw_root, exist_ok=True)
    zip_path = os.path.join(raw_root, "maestro-v3.0.0-midi.zip")
    urllib.request.urlretrieve("https://storage.googleapis.com/magentadata/datasets/maestro/v3.0.0/maestro-v3.0.0-midi.zip", zip_path)
    print("Extracting...")
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(raw_root)
    print("Done!")

data_dir = maestro_dir if os.path.exists(os.path.join(maestro_dir, metadata_file)) else fallback_dir
metadata_path = os.path.join(data_dir, metadata_file)
metadata = pd.read_csv(metadata_path)

train_meta = metadata[metadata["split"] == "train"]
val_meta = metadata[metadata["split"] == "validation"]
test_meta = metadata[metadata["split"] == "test"]

os.makedirs("data/train_test_split", exist_ok=True)
train_meta.to_csv("data/train_test_split/train.csv", index=False)
val_meta.to_csv("data/train_test_split/val.csv", index=False)
test_meta.to_csv("data/train_test_split/test.csv", index=False)

print(f"Training files: {len(train_meta)}, Val files: {len(val_meta)}, Test files: {len(test_meta)}")

Extracting...
Done!
Training files: 962, Val files: 137, Test files: 177


## Part 1: Piano-Roll Representation (For Task 1 & 2)

In [4]:
FS = 16 # Frames per second
SEQ_LEN = 128 # 8 seconds of music
PITCH_RANGE = (21, 109) # 88 piano keys (A0 to C8)
SPARSITY_THRESHOLD = 0.02 # Dismiss windows with <2% active cells

def process_piano_roll(df):
    valid_segments = []
    for idx, row in tqdm(df.iterrows(), total=len(df)):
        midi_file = os.path.join(data_dir, row["midi_filename"])
        try:
            pm = pretty_midi.PrettyMIDI(midi_file)
            # 1. Extract piano-roll (fs=16)
            proll = pm.get_piano_roll(fs=FS)[PITCH_RANGE[0]:PITCH_RANGE[1], :]
            # 2. Binarize 
            proll = (proll > 0).astype(np.float32)
            # Transpose to (Time, Pitches)
            proll = proll.T
            
            # 3. Segment into windows
            n_segments = proll.shape[0] // SEQ_LEN
            for i in range(n_segments):
                win = proll[i*SEQ_LEN : (i+1)*SEQ_LEN, :]
                # 4. Sparsity Filtering
                if np.mean(win) > SPARSITY_THRESHOLD:
                    valid_segments.append(win)
        except Exception:
            continue
    return np.array(valid_segments)

processed_rolls_dir = os.path.join("data", "processed", "rolls")
legacy_rolls_dir = os.path.join("data", "processed_rolls")
os.makedirs(processed_rolls_dir, exist_ok=True)
os.makedirs(legacy_rolls_dir, exist_ok=True)

train_rolls = process_piano_roll(train_meta)
val_rolls = process_piano_roll(val_meta)
test_rolls = process_piano_roll(test_meta)

np.save(os.path.join(processed_rolls_dir, "train.npy"), train_rolls)
np.save(os.path.join(processed_rolls_dir, "val.npy"), val_rolls)
np.save(os.path.join(processed_rolls_dir, "test.npy"), test_rolls)
np.save(os.path.join(legacy_rolls_dir, "train.npy"), train_rolls)
np.save(os.path.join(legacy_rolls_dir, "val.npy"), val_rolls)
np.save(os.path.join(legacy_rolls_dir, "test.npy"), test_rolls)

# Calculate negative to positive ratio for BCEWithLogitsLoss pos_weight later
num_positive = np.sum(train_rolls == 1)
num_negative = np.sum(train_rolls == 0)
pos_weight = num_negative / max(num_positive, 1)
with open(os.path.join(processed_rolls_dir, "pos_weight.txt"), "w") as f:
    f.write(str(pos_weight))
with open(os.path.join(legacy_rolls_dir, "pos_weight.txt"), "w") as f:
    f.write(str(pos_weight))
print(f"Saved {len(train_rolls)} train, {len(val_rolls)} val, {len(test_rolls)} test rolls. Suggested pos_weight: {pos_weight:.2f}")

100%|██████████| 177/177 [00:29<00:00,  5.97it/s]


Saved 62689 train, 7876 val, 7792 test rolls. Suggested pos_weight: 16.19


## Part 2: Token Representation (For Task 3 & 4)

In [6]:
config = TokenizerConfig(num_velocities=32, use_chords=False, use_programs=False)
tokenizer = REMI(config)

def normalize_token_ids(tokens):
    if hasattr(tokens, "ids"):
        ids = tokens.ids
        if isinstance(ids, list) and ids and isinstance(ids[0], list):
            return ids
        return [ids]
    if isinstance(tokens, list):
        out = []
        for t in tokens:
            if hasattr(t, "ids"):
                ids = t.ids
                if isinstance(ids, list) and ids and isinstance(ids[0], list):
                    out.extend(ids)
                else:
                    out.append(ids)
        return out
    return []

def process_tokens(df, min_len=50):
    token_sequences = []
    for idx, row in tqdm(df.iterrows(), total=len(df)):
        midi_file = os.path.join(data_dir, row["midi_filename"])
        try:
            tokens = tokenizer(midi_file)
            for seq in normalize_token_ids(tokens):
                if len(seq) >= min_len:
                    token_sequences.append(seq)
        except:
            continue
    return token_sequences

processed_tokens_dir = os.path.join("data", "processed", "tokens")
legacy_tokens_dir = os.path.join("data", "processed_tokens")
os.makedirs(processed_tokens_dir, exist_ok=True)
os.makedirs(legacy_tokens_dir, exist_ok=True)

train_tokens = process_tokens(train_meta)
val_tokens = process_tokens(val_meta)
test_tokens = process_tokens(test_meta)

# Save as list of arrays
np.save(os.path.join(processed_tokens_dir, "train.npy"), np.array(train_tokens, dtype=object))
np.save(os.path.join(processed_tokens_dir, "val.npy"), np.array(val_tokens, dtype=object))
np.save(os.path.join(processed_tokens_dir, "test.npy"), np.array(test_tokens, dtype=object))
np.save(os.path.join(legacy_tokens_dir, "train.npy"), np.array(train_tokens, dtype=object))
np.save(os.path.join(legacy_tokens_dir, "val.npy"), np.array(val_tokens, dtype=object))
np.save(os.path.join(legacy_tokens_dir, "test.npy"), np.array(test_tokens, dtype=object))
print(f"Saved {len(train_tokens)} train, {len(val_tokens)} val, {len(test_tokens)} test sequences for Transformer.")

  0%|          | 0/962 [00:00<?, ?it/s]

100%|██████████| 177/177 [00:06<00:00, 25.36it/s]


Saved 962 train, 137 val, 177 test sequences for Transformer.
